# 📘 Boosting – Study Notes
---

## 1. What is Boosting?

- An **ensemble technique** — combines several weak models to make one strong model
- Starts from a weak decision-maker and keeps building models on top of each other
- Final prediction = **weighted sum of all weak models**
- Weights are assigned based on each model's **individual performance**
- Parameters are calculated **stagewise** — each new tree learns from the mistakes of the previous one

**General objective function:**
$$obj = L + \Omega$$
- `L` = Loss function (controls predictive power)
- `Ω` = Regularization (controls model simplicity)

## 2. Why Decision Trees as Weak Classifiers?

A **weak classifier** = one that is only slightly better than random guessing.

Trees are chosen because they:
- Scale computationally well
- Handle missing values
- Are robust to outliers
- Don't require feature scaling
- Can handle both numerical and categorical features
- Are easy to interpret (when small)

**Weakness of trees:**
- Can't extract linear combinations of features
- High variance → low predictive power alone

Boosting fixes the high variance problem by **averaging results across many trees**.

## 3. AdaBoost (Adaptive Boosting)

**Core idea:** Misclassified samples get **higher weights** in the next round so the next tree focuses on them.
AdaBoost (Adaptive Boosting) assigns equal weights to all training samples initially and iteratively adjusts these weights by focusing more on misclassified data points for the next model. It effectively reduces bias and variance making it useful for classification tasks 
**Algorithm steps:**

1. Assign equal weight to all N samples: $w_i = \frac{1}{N}$
2. For each tree `m`:
   - Fit a weak classifier $G_m(x)$ using current weights
   - Calculate **weighted error:**
     $$err_m = \frac{\sum w_i \cdot \mathbb{1}(y_i \neq G_m(x))}{\sum w_i}$$
   - Calculate **contribution** of this tree:
     $$\alpha_m = \frac{1}{2} \log\left(\frac{1 - err_m}{err_m}\right)$$
   - Update weights:
     - Incorrectly classified → **increase weight:** $w_i \times e^{+\alpha_m}$
     - Correctly classified → **decrease weight:** $w_i \times e^{-\alpha_m}$
   - Normalize weights so they sum to 1
3. Final output:
   - Regression: $G(x) = \arg\max \left[\sum_{m=1}^{M} \alpha_m G_m(x)\right]$
   - Classification: $G(x) = \text{sigmoid} \left[\sum_{m=1}^{M} \alpha_m G_m(x)\right]$

## 4. AdaBoost – Worked Example (Heart Disease)

- Dataset: 8 rows, features are Chest Pain, Blocked Arteries, Body Weight
- Initial sample weight for each row = 1/8 (all equally important)

**Step 1 – Select root node:**
- Calculate Gini Index for each feature stump:
  - Chest Pain → GI = 0.47
  - Blocked Arteries → GI = 0.5
  - Body Weight → GI = 0.2 ✅ (lowest → selected)

**Step 2 – Calculate contribution:**
- Body Weight stump misclassifies 1 out of 8 → error = 1/8
- Contribution = $\frac{1}{2} \log\left(\frac{1 - 0.125}{0.125}\right) = 0.97$

**Step 3 – Update weights:**
- Incorrect sample: $\frac{1}{8} \times e^{0.97} = 0.33$
- Correct samples: $\frac{1}{8} \times e^{-0.97} = 0.05$

**Step 4 – Normalize** → divide all new weights by their sum (0.68)

**Final decision:** Sum contributions of all trees voting for each class → class with higher total wins.

## 5. Gradient Boosted Trees

- Uses decision trees as estimators
- Works with **any loss function** (regression, classification, risk modelling, etc.)
- Calculates the **gradient of the error** and fits each new tree to approximate it
- **AdaBoost is a special case** of Gradient Boosted Trees (uses exponential loss)

**Algorithm steps:**
1. Start with the **average** of the target column as the first prediction (minimizes initial error)
2. Calculate **pseudo residuals** = Actual − Predicted
   $$\text{residual} = \frac{\delta L(y_i, f(x_i))}{\delta f(x_i)}$$
3. Build a tree to **predict the residuals** (not the actual target)
4. Update prediction:
   $$F_1(x) = F_0(x) + \nu \cdot \text{residual}$$
   where $\nu$ = learning rate
5. Repeat until residuals stop decreasing

**Final prediction:**
$$\text{Final} = \text{First prediction} + \nu \times r_1 + \nu \times r_2 + \ldots$$

## 6. Gradient Boosting – Worked Example (Weight Prediction)

- Dataset: 6 people with weights [88, 76, 56, 73, 77, 57]

**Step 1:** Average = (88+76+56+73+77+57) / 6 = **71.2** → first prediction for all rows

**Step 2:** Compute residuals (e.g., 88 − 71.2 = 16.8, 76 − 71.2 = 4.8, ...)

**Step 3:** Build a tree to predict residuals. If a leaf has multiple residuals, use their average.

**Step 4:** Update with learning rate = 0.1:
$$\text{New value} = 71.2 + 0.1 \times 16.8 = 72.9 \text{ (for first row)}$$

**Repeat** until residuals don't improve significantly.

## 7. XGBoost (Extreme Gradient Boosting)
 XGBoost is an optimized version of Gradient Boosting that uses regularization to prevent overfitting. It is faster, efficient and supports handling both numerical and categorical variables.
- Improves Gradient Boosting with **better regularization**
- Developed by Tianqi Chen in C++; has Python, R, Julia interfaces
- Does **not use Gini or Entropy** — uses **gradient and hessian** instead

**Objective function:**
$$obj(\theta) = \sum_{i=1}^{n} l(y_i - \hat{y}_i) + \sum_{j=1}^{J} \Omega(f_j)$$

**Regularization term:**
$$\Omega(f) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2$$
- $\gamma T$ → penalizes number of leaves (controls tree size)
- $\lambda \sum w_j^2$ → penalizes large leaf scores

**Hessian:**
- Second-order derivative of the loss function
- For regression → number of residuals
- Used alongside gradient for smarter, more stable splits

**XGBoost algorithm steps:**
1. Start with one leaf
2. Compute **similarity score** for each node:
   $$\text{Similarity} = \frac{\text{Gradient}^2}{\text{Hessian} + \lambda}$$
3. Compute **gain** for each split:
   $$\text{Gain} = \text{Left similarity} + \text{Right similarity} - \text{Root similarity}$$
4. **Prune** using $\gamma$: if $\text{Gain} - \gamma < 0$ → remove that branch
5. Update prediction:
   $$\text{New Value} = \text{Old Value} + \eta \times \text{prediction}$$
   where $\eta$ = learning rate

## 8. XGBoost Key Hyperparameters

| Parameter | What it does |
|---|---|
| `learning_rate` | How much each tree contributes (smaller = more trees needed but more robust) |
| `max_depth` | Max depth of each tree (controls complexity) |
| `n_estimators` | Number of trees to build |
| `objective` | Loss function (e.g., `'binary:logistic'` for classification) |
| `lambda` (reg_lambda) | L2 regularization on leaf weights |
| `gamma` | Min gain required to make a split (pruning control) |

## 10. AdaBoost vs Gradient Boosting vs XGBoost

| Feature | AdaBoost | Gradient Boosting | XGBoost |
|---|---|---|---|
| Focuses on | Misclassified samples (via weights) | Residuals (errors) | Residuals + Regularization |
| Loss function | Exponential | Any (pluggable) | Any + built-in regularization |
| Tree type | Decision stumps (depth=1) | Full trees | Full trees |
| Regularization | None | None | Yes (γ and λ) |
| Speed | Moderate | Slow | Fast (parallel processing) |
| Overfitting control | Weak | Moderate | Strong |

In [ ]:
##Support Vector Machine (SVM) is a supervised machine learning algorithm used for classification and regression tasks.
It tries to find the best boundary known as hyperplane that separates different classes in the data.
It is useful when you want to do binary classification like spam vs. not spam or cat vs. dog.

The main goal of SVM is to maximize the margin between the two classes.
The larger the margin the better the model performs on new and unseen data.
The key idea behind the SVM algorithm is to find the hyperplane that best separates two classes by maximizing the margin between them.
This margin is the distance from the hyperplane to the nearest data points (support vectors) on each side.
